# KELOMPOK 6 PBA

ANGGOTA KELOMPOK:
- Abit Ahmad Oktarian 	- 122450042 -	Danielle024
- Fadhil Fitra Wijaya 	- 122450082 -	epwefwe
- Dhafin Razaqa 	- 122450133 -	DhafinRazaqaLuthfi

# 🎮 Steam Review Sentiment Analysis - NLP Pipeline
Proyek ini bertujuan untuk membangun model Machine Learning (NLP) yang dapat mendeteksi sentimen dari ulasan game di Steam (Positif/Negatif).

## Langkah 1: Import Library & Konfigurasi
Bagian ini berisi semua pengaturan dasar, folder penyimpan, kamus *slang* (bahasa gaul), dan pemetaan *leetspeak* yang akan digunakan di seluruh proses.

In [ ]:
import os
import re
import pandas as pd
import warnings
import logging

# Matikan warning agar notebook terlihat rapi
warnings.filterwarnings("ignore")
os.environ["LGBM_WARNING"] = "0"
logging.getLogger("lightgbm").setLevel(logging.ERROR)

# --- PATH & FOLDERS ---
BASE_DIR = os.path.abspath("")
DATA_DIR = os.path.join(BASE_DIR, "data")
MODEL_DIR = os.path.join(BASE_DIR, "models")
RAW_CSV = os.path.join(DATA_DIR, "dataset.csv")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# --- DATASET CONFIG ---
TEXT_COL = "review_text"
LABEL_COL = "review_score"
SESSION_ID = 42
TRAIN_SIZE = 0.8
N_TOP_MODELS = 5

# --- KAMUS NLP ---
LEETSPEAK_MAP = {
    "0": "o", "1": "i", "2": "z", "3": "e", "4": "a",
    "5": "s", "6": "g", "7": "t", "8": "b", "9": "g", "@": "a",
}

SLANG_DICT = {
    "anj": "anjing", "gblk": "goblok", "bgst": "bangsat", "kntl": "kontol", 
    "gw": "gue", "lu": "lo", "ga": "tidak", "gak": "tidak", "kyk": "kayak", 
    "bgt": "banget", "udh": "sudah", "yg": "yang", "tp": "tapi", "krn": "karena", 
    "jgn": "jangan", "aja": "saja", "noob": "pemula", "gg": "good game", 
    "ez": "easy", "lag": "lag", "dc": "disconnect", "bcs": "karena"
}
print("✅ Konfigurasi dan Kamus NLP berhasil dimuat!")

## Langkah 2: Mengunduh Dataset
Mengunduh dataset dari Kaggle secara otomatis menggunakan library `kagglehub`. Jika file sudah ada di folder `data/`, proses ini akan dilewati (skip).

import shutil
import glob
import kagglehub

def download_dataset():
    if os.path.exists(RAW_CSV):
        print(f"✅ Dataset sudah ada: {RAW_CSV}")
        return RAW_CSV

    print("📥 Mendownload dataset dari Kaggle...")
    try:
        download_path = kagglehub.dataset_download("jprestiliano/indonesian-chat-dataset")
        csv_files = glob.glob(os.path.join(download_path, "**", "*.csv"), recursive=True)

        if csv_files:
            shutil.copy2(csv_files[0], RAW_CSV)
            print(f"✅ Dataset disalin ke: {RAW_CSV}")
            return RAW_CSV
        else:
            print("❌ File CSV tidak ditemukan.")
    except Exception as e:
        print(f"⚠️ Gagal mendownload: {e}")

download_dataset()

## Langkah 3: Preprocessing & Pembersihan Teks
Tahap ini sangat krusial dalam NLP. Kita akan mengubah teks menjadi huruf kecil (lowercase), menghapus URL/mention, menormalkan angka yang dijadikan huruf (*leetspeak*), mengubah kata gaul menjadi baku, dan membatasi data sebanyak **30.000 baris** agar proses komputasi lebih ringan.

In [ ]:
def normalize_leetspeak(text):
    result = []
    for i, char in enumerate(text):
        if char in LEETSPEAK_MAP:
            prev_is_alpha = (i > 0 and text[i - 1].isalpha())
            next_is_alpha = (i < len(text) - 1 and text[i + 1].isalpha())
            if prev_is_alpha or next_is_alpha:
                result.append(LEETSPEAK_MAP[char])
            else:
                result.append(char)
        else:
            result.append(char)
    return "".join(result)

def clean_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = normalize_leetspeak(text)
    words = text.split()
    text = " ".join([SLANG_DICT.get(w, w) for w in words])
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("--- Membaca dan membersihkan dataset (Max 30k baris) ---")
df = pd.read_csv(RAW_CSV, nrows=30000)

# Gunakan fallback jika nama kolom berbeda
if TEXT_COL not in df.columns:
    df = df.rename(columns={"app_name": TEXT_COL, "review_score": LABEL_COL}) # Sesuaikan fallback Anda

df = df.dropna(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)
df["cleaned_text"] = df[TEXT_COL].apply(clean_text)
df = df[df["cleaned_text"].str.len() > 0].reset_index(drop=True)

print(f"✅ Selesai! Jumlah baris bersih: {len(df)}")
display(df[[TEXT_COL, "cleaned_text", LABEL_COL]].head())

## Langkah 4: AutoML dengan PyCaret
Di sini kita menyerahkan data yang sudah bersih ke PyCaret. PyCaret akan secara otomatis mengubah teks menjadi angka (TF-IDF), membandingkan belasan algoritma Machine Learning sekaligus, memilih yang terbaik (LightGBM/Logistic Regression), lalu menyimpannya sebagai file `.pkl`.

In [ ]:
from pycaret.classification import setup, compare_models, finalize_model, save_model

print("⚙️ Menginisialisasi PyCaret...")
df_model = df[["cleaned_text", LABEL_COL]].copy()

# Tips: html=True sangat bagus untuk Jupyter Notebook agar tabelnya rapi
s = setup(
    data=df_model,
    target=LABEL_COL,
    text_features=["cleaned_text"],
    session_id=SESSION_ID,
    train_size=TRAIN_SIZE,
    verbose=True,
    html=True, 
    use_gpu=False,
    fold=3
)

print("🏟️ Memulai proses perbandingan model (AutoML)...")
best_model = compare_models(sort="F1", n_select=1, exclude=['xgboost'])

print("💾 Memfinalisasi dan menyimpan model...")
final_model = finalize_model(best_model)
save_model(final_model, os.path.join(MODEL_DIR, "nlp_pipeline_final"))
print("✅ Model sukses disimpan!")

## Langkah 5: Deployment Demo dengan Gradio
Langkah terakhir adalah memuat model `.pkl` yang baru saja dibuat, lalu memasukkannya ke dalam antarmuka web interaktif menggunakan Gradio. Cell ini akan menghasilkan UI bertema "Steam" langsung di bawah cell ini!

In [ ]:
import gradio as gr
from pycaret.classification import load_model, predict_model

print("Memuat model NLP untuk Demo...")
saved_model = load_model(os.path.join(MODEL_DIR, "nlp_pipeline_final"))

def predict_sentiment(teks_review):
    if not teks_review.strip(): return "Harap masukkan teks review."
    try:
        teks_bersih = clean_text(teks_review)
        df_input = pd.DataFrame({'cleaned_text': [teks_bersih]})
        prediksi = predict_model(saved_model, data=df_input)
        
        hasil = prediksi['prediction_label'].iloc[0] if 'prediction_label' in prediksi.columns else prediksi['Label'].iloc[0]
        return "👍 POSITIF (Recommended)" if hasil == 1 else "👎 NEGATIF (Not Recommended)"
    except Exception as e:
        return f"Error: {str(e)}"

# Tema Steam Custom
bg_color = "#171a21"
primary_accent = "#66c0f4"

custom_css = f"""
    body, .gradio-container {{ background-color: {bg_color}; font-family: 'Trebuchet MS', sans-serif; }}
    .gr-button-primary {{ background-color: {primary_accent} !important; color: black !important; font-weight: bold; }}
"""

gamer_theme = gr.themes.Default(primary_hue="blue").set(
    body_background_fill=bg_color,
    body_text_color="#c7d5e0",
    input_background_fill="#101214",
)

with gr.Blocks(theme=gamer_theme, css=custom_css) as demo:
    gr.Markdown("# 🎮 Steam Review Sentiment Analyzer")
    gr.Markdown("Ketik review game di bawah ini untuk dideteksi oleh AI.")
    
    with gr.Row():
        input_text = gr.Textbox(lines=3, label="Review Game", placeholder="Tulis review di sini...")
        output_text = gr.Textbox(label="Hasil Analisis AI", interactive=False)
        
    predict_btn = gr.Button("Analisis Sentimen", variant="primary")
    predict_btn.click(fn=predict_sentiment, inputs=input_text, outputs=output_text)

# Menjalankan Gradio INLINE di dalam Jupyter Notebook
demo.launch(inline=True, share=False)